In [ ]:
# --- 1. INSTALAÇÃO ---
print("Instalando bibliotecas...")
!pip install geopandas folium shapely gspread google-auth google-auth-oauthlib google-auth-httplib2 ipynbname unidecode fuzzywuzzy python-Levenshtein -q
print("Bibliotecas instaladas.")

# --- 2. IMPORTAÇÕES ---
import os
import time
import datetime
import math
import requests
import pandas as pd
import geopandas as gpd
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from io import BytesIO
from shapely.geometry import LineString, MultiLineString, mapping, Polygon
from shapely.ops import polygonize, nearest_points, transform
from unidecode import unidecode
from fuzzywuzzy import fuzz
import folium
from branca.element import MacroElement
from jinja2 import Template
from google.colab import userdata, drive, auth as colab_auth
from google.auth import default as google_auth_default
import gspread
import urllib3
import numpy as np
import warnings
from IPython.display import display

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore")

# --- 3. CONFIGURAÇÃO ---
try:
    drive.mount('/content/drive')
    base_path_secret = userdata.get('DRIVE_SAVE_PATH') if 'DRIVE_SAVE_PATH' in userdata else "/content/drive/MyDrive/Colab_Pipelines/Maia_Hidro_Audit/"
    DRIVE_SAVE_PATH = base_path_secret
    if not os.path.exists(DRIVE_SAVE_PATH): os.makedirs(DRIVE_SAVE_PATH)
except:
    DRIVE_SAVE_PATH = "/content/"

# Secrets & Email Config
try:
    GMAIL_USER = userdata.get('GMAIL_USER')
    GMAIL_APP_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
    email_list_raw = userdata.get('DESTINATION_EMAIL_LIST')
    if "," in email_list_raw:
        DESTINATION_EMAIL_LIST = [e.strip() for e in email_list_raw.split(',')]
    else:
        DESTINATION_EMAIL_LIST = [email_list_raw.strip()]
except Exception as e:
    print(f"⚠️ Aviso: Secrets de email não configurados corretamente. ({e})")
    GMAIL_USER = None
    DESTINATION_EMAIL_LIST = []

# URLs
MAIA_BOUNDARY_URL = "https://baze.cm-maia.pt/BaZe/api/api4gj.php?nome=limconcvm"
OFFICIAL_DATA_URL = "https://geoserver.cm-maia.pt/geoserver/amp/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=amp:Linhas%20de%20agua&maxFeatures=5000&outputFormat=application%2Fjson"

HTML_MAP_FILENAME = os.path.join(DRIVE_SAVE_PATH, f"auditoria_hidro_{datetime.date.today().isoformat()}.html")
GOOGLE_SHEET_NAME = f"Auditoria_Hidro_OSM_{datetime.date.today().isoformat()}"

# Servidores OSM
OVERPASS_URLS = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass-api.de/api/interpreter"
]
API_TIMEOUT = 180
MATCH_THRESHOLD = 50.0 # Metros de tolerância para considerar que é o mesmo rio

# --- ESTILOS VISUAIS ---
STYLE_CONFIG = {
    "Total": {"folium": "blue", "hex": "#0000FF", "icon": "tint", "label": "Match Confirmado (Nome + Local)"},
    "ConflitoNome": { "folium": "orange", "hex": "#FFA500", "icon": "font", "label": "Conflito de Nomes" },
    "Apenas": {"folium": "red", "hex": "#FF0000", "icon": "times", "label": "Apenas no WFS (Falta no Mapa)"},
    "Default": {"folium": "gray", "hex": "gray", "icon": "info", "label": "Indefinido"}
}

# --- 4. FUNÇÕES AUXILIARES & PROFILING ---
profiling_log = []
script_start_time = time.time()

def start_timer(): return time.time()

def log_time(task, start_t):
    elapsed_proc = time.time() - start_t
    elapsed_watch = time.time() - script_start_time
    print(f"[{task}] {elapsed_proc:.2f}s")
    profiling_log.append({"task": task, "proc_time": elapsed_proc, "watch_time": elapsed_watch})

def normalize_text(text):
    if not text: return ""
    return unidecode(str(text)).strip().lower()

def get_safe_bounds(poly):
    try:
        minx, miny, maxx, maxy = poly.bounds
        if np.isnan(minx): raise ValueError("Bounds NaN")
        return minx, miny, maxx, maxy
    except:
        return -8.691, 41.185, -8.530, 41.309

context_vars = {'sheet_url': '#', 'date': str(datetime.date.today())}

# --- 5. CARREGAMENTO DE DADOS ---

def load_maia_boundary(url):
    s = start_timer()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            r = requests.get(url, verify=False, timeout=30)
            gdf = gpd.read_file(BytesIO(r.content))
            if gdf.empty: return None
            if gdf.crs and gdf.crs.to_string() != "EPSG:4326": gdf = gdf.to_crs("EPSG:4326")

            geom = gdf.geometry.unary_union
            if geom.geom_type in ['LineString', 'MultiLineString']:
                polys = list(polygonize(geom))
                if polys: geom = max(polys, key=lambda a: a.area)
                else: geom = geom.convex_hull
            elif not geom.is_valid:
                geom = geom.buffer(0)

            log_time("Limite Maia", s)
            return geom
    except Exception as e:
        print(f"Erro limite: {e}")
        return None

def load_wfs_hidro(url):
    s = start_timer()
    print("A carregar WFS Hidrografia...")
    try:
        r = requests.get(url, verify=False, timeout=60)
        gdf = gpd.read_file(BytesIO(r.content))

        if gdf.crs is None: gdf.set_crs("EPSG:3763", inplace=True)

        def get_name(row):
            if row.get('04_DESIGNC') and str(row.get('04_DESIGNC')).strip() != '': return str(row.get('04_DESIGNC'))
            if row.get('03_BH'): return str(row.get('03_BH'))
            return "Linha de Água s/ Nome"

        gdf['Name'] = gdf.apply(get_name, axis=1)
        gdf['Name_Clean'] = gdf['Name'].apply(normalize_text)

        gdf = gdf.to_crs("EPSG:3763")
        gdf['geometry_map'] = gdf.to_crs("EPSG:4326").geometry

        log_time(f"WFS Hidro ({len(gdf)} segmentos)", s)
        return gdf
    except Exception as e:
        print(f"Erro WFS: {e}")
        return gpd.GeoDataFrame()

def query_osm_waterways(polygon, timeout_seconds):
    s = start_timer()
    print("A carregar OSM (Waterways)...")
    if polygon is None: return gpd.GeoDataFrame()

    minx, miny, maxx, maxy = get_safe_bounds(polygon)

    query = f"""
    [out:json][timeout:{timeout_seconds}];
    (
      way["waterway"~"^(river|stream|drain|canal|brook)$"]({miny-0.01},{minx-0.01},{maxy+0.01},{maxx+0.01});
    );
    out geom tags;
    """

    for url in OVERPASS_URLS:
        try:
            r = requests.post(url, data={"data": query}, timeout=timeout_seconds)
            if r.status_code == 200:
                elements = r.json().get("elements", [])
                features = []
                for e in elements:
                    if 'geometry' in e:
                        coords = [(pt['lon'], pt['lat']) for pt in e['geometry']]
                        if len(coords) >= 2:
                            features.append({
                                'Name': e.get('tags', {}).get('name', 'Sem Nome'),
                                'Name_Clean': normalize_text(e.get('tags', {}).get('name', '')),
                                'OSM_ID': e['id'],
                                'geometry': LineString(coords)
                            })

                if features:
                    gdf = gpd.GeoDataFrame(features, crs="EPSG:4326")
                    gdf = gdf[gdf.intersects(polygon)].copy()
                    log_time(f"OSM Raw ({len(gdf)})", s)
                    return gdf.to_crs("EPSG:3763")
        except: continue

    print("❌ Falha Total no OSM.")
    return gpd.GeoDataFrame()

# --- 6. RECONCILIAÇÃO (Linha a Linha) ---
def reconcile_lines(wfs, osm):
    s = start_timer()
    print("A reconciliar (WFS vs OSM)...")
    results = []

    if not osm.empty:
        osm_sindex = osm.sindex

    for idx, row in wfs.iterrows():
        wfs_geom = row.geometry
        match_name = "N/A"
        dist_m = 9999
        osm_id = ""
        status = "Apenas"
        code = "Apenas"

        if not osm.empty:
            try:
                possible_matches_index = list(osm_sindex.intersection(wfs_geom.buffer(MATCH_THRESHOLD).bounds))
                possible_matches = osm.iloc[possible_matches_index]

                if not possible_matches.empty:
                    possible_matches = possible_matches.copy()
                    possible_matches['dist'] = possible_matches.distance(wfs_geom)

                    nearby = possible_matches[possible_matches['dist'] <= MATCH_THRESHOLD].sort_values('dist')

                    if not nearby.empty:
                        best = nearby.iloc[0]
                        dist_m = best['dist']
                        osm_id = best['OSM_ID']
                        match_name = best['Name']

                        if fuzz.token_set_ratio(row['Name_Clean'], best['Name_Clean']) > 70:
                            code, status = "Total", "Match Confirmado"
                        else:
                            code, status = "ConflitoNome", "Conflito de Nomes"
            except: pass

        geo_map = row['geometry_map']
        mid_point = None
        try:
             if geo_map.geom_type == 'MultiLineString':
                 mid_point = geo_map.geoms[0].interpolate(0.5, normalized=True)
             else:
                 mid_point = geo_map.interpolate(0.5, normalized=True)
        except: pass

        results.append({
            'Nome_Oficial': row['Name'],
            'Status': status, 'Code': code,
            'Nome_OSM': match_name, 'OSM_ID': osm_id,
            'Dist_OSM': round(dist_m, 1) if dist_m != 9999 else "-",
            'Lat': mid_point.y if mid_point else None,
            'Lon': mid_point.x if mid_point else None,
            'geometry': geo_map
        })

    log_time("Reconciliação Hidro", s)
    return gpd.GeoDataFrame(results, crs="EPSG:4326")

# --- 7. MAPA ---
class MapLegend(MacroElement):
    _template = Template(u"""
    {% macro html(this, kwargs) %}
    <div style="position: fixed; bottom: 30px; right: 30px; width: 340px; z-index:9999; border: 2px solid rgba(0,0,0,0.2); border-radius: 8px; background-color: rgba(255, 255, 255, 0.95); padding: 15px; font-family: 'Segoe UI', Arial, sans-serif; font-size: 13px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

    <h4 style="margin-top:0; margin-bottom: 12px; font-size: 16px; color: #333; border-bottom: 1px solid #ddd; padding-bottom: 8px;"><b>{{ this.title }}</b></h4>

    <table style="width:100%; border-collapse: collapse;">
        {% for label, config in this.items %}
        <tr style="margin-bottom: 8px; border-bottom: 1px solid #f0f0f0;">
            <td style="width: 30px; text-align: center; padding: 6px;"><i class="fa fa-{{ config.icon }}" style="color: {{ config.hex }}; font-size: 16px;"></i></td>
            <td style="padding: 6px; color: #444;">{{ label }}</td>
        </tr>
        {% endfor %}
    </table>

    <div style="margin-top: 10px; border-top: 1px solid #eee; padding-top: 8px; text-align: center;">
        <small style="color:#555; font-weight:bold; display:block; margin-bottom:4px;">Fontes de Dados</small>
        <div style="font-size: 12px; font-weight: bold; color: #444;">
            WFS (CM Maia)
            <span style="color:#ccc; margin:0 4px;">|</span>
            <a href="https://www.openstreetmap.org/" target="_blank" style="color:#0078A8; text-decoration:none;">OSM</a>
        </div>
    </div>

    </div>
    {% endmacro %}
    """)
    def __init__(self, title, items):
        super(MapLegend, self).__init__()
        self._name = 'MapLegend'
        self.title = title
        self.items = items

def create_map_hidro(gdf, poly_geom, filename):
    print("Gerando Mapa...")
    m = folium.Map(location=[41.23, -8.62], zoom_start=13, tiles="CartoDB positron")

    if poly_geom is not None:
        try:

            folium.GeoJson(
                data=mapping(poly_geom.simplify(0.0001)),
                name="Limite Maia",
                style_function=lambda x: {'color': '#000000', 'weight': 3, 'fillOpacity': 0, 'dashArray': '5, 5'}
            ).add_to(m)
        except Exception as e:
            print(f"⚠️ Aviso: Não foi possível desenhar o limite ({e})")

    def get_coords_2d(geom):
        if geom.geom_type == 'LineString':
            return [[(p[1], p[0]) for p in geom.coords]]
        elif geom.geom_type == 'MultiLineString':
            lines = []
            for g in geom.geoms:
                lines.append([(p[1], p[0]) for p in g.coords])
            return lines
        return []

    for _, r in gdf.iterrows():
        code = r['Code']
        config = STYLE_CONFIG.get(code, STYLE_CONFIG['Default'])
        col = config['hex']

        popup_html = f"""
        <div style='font-family: "Segoe UI", sans-serif; width: 320px; font-size: 13px;'>
            <div style='background-color: {col}; color: white; padding: 6px 10px; border-radius: 4px;'><b>{r['Status']}</b></div>
            <div style="margin-top: 8px;"><b style='color:#333; font-size:15px;'>🌊 {r['Nome_Oficial']}</b></div>
            <hr>
            <div style="margin-top:5px;">
               <b>🌍 OSM:</b> {r['Nome_OSM']} (Dist: {r['Dist_OSM']}m)
            </div>
        </div>"""

        lines_coords = get_coords_2d(r['geometry'])

        for line in lines_coords:
            folium.PolyLine(
                line,
                color=col,
                weight=4,
                opacity=0.8,
                popup=folium.Popup(popup_html, max_width=350)
            ).add_to(m)

    items = [(v['label'], v) for k, v in STYLE_CONFIG.items() if k != "Default"]
    m.add_child(MapLegend("Auditoria Hidrografia", items))
    m.save(filename)
    return m

def save_sheet(df, name):
    try:
        colab_auth.authenticate_user(); creds, _ = google_auth_default(); gc = gspread.authorize(creds)
        try: sh = gc.open(name)
        except: sh = gc.create(name)
        ws = sh.get_worksheet(0); ws.clear()

        df_save = df.drop(columns=['geometry', 'geometry_map'], errors='ignore')
        ws.update([df_save.columns.values.tolist()] + df_save.fillna('').astype(str).values.tolist(), 'A1')

        for e in DESTINATION_EMAIL_LIST:
            try: sh.share(e, perm_type='user', role='writer')
            except: pass
        return f"https://docs.google.com/spreadsheets/d/{sh.id}"
    except: return "#"

# --- 8. FUNÇÃO DE EMAIL ---
def send_execution_email(sheet_url, df_res):
    if not GMAIL_USER or not DESTINATION_EMAIL_LIST:
        print("⏭️ Email não enviado (Credenciais em falta).")
        return

    print("📧 A enviar email de relatório...")

    # Stats
    stats = df_res['Status'].value_counts().to_dict() if not df_res.empty else {}
    stats_html = "<ul>" + "".join([f"<li>{k}: {v}</li>" for k, v in stats.items()]) + "</ul>"

    log_rows = ""
    for entry in profiling_log:
        log_rows += f"""
        <tr>
            <td style="border: 1px solid black; padding: 4px; font-family: monospace;">{entry['task']}</td>
            <td style="border: 1px solid black; padding: 4px; text-align: right;">{entry['proc_time']:.2f}s</td>
        </tr>
        """

    total_time = time.time() - script_start_time

    html_body = f"""
    <html>
    <body style="font-family: Arial, sans-serif;">
        <h2 style="color: #0000FF;">Pipeline Hidrografia - Execution Log</h2>
        <p>O processo de auditoria de Linhas de Água (WFS vs OSM) foi concluído.</p>

        <h3>Resumo Estatístico</h3>
        {stats_html}

        <p><b>🔗 Google Sheet Resultado:</b> <a href="{sheet_url}">{sheet_url}</a></p>

        <h3 style="margin-top:20px;">Execution Timing</h3>
        <table style="border-collapse: collapse; width: 100%; font-size: 12px; border: 2px solid black;">
            <thead style="background-color: #f2f2f2;">
                <tr><th style="border: 1px solid black; padding: 6px;">Task</th><th style="border: 1px solid black; padding: 6px;">Time</th></tr>
            </thead>
            <tbody>{log_rows}</tbody>
            <tfoot><tr><td><b>Total</b></td><td style="text-align:right;"><b>{total_time:.2f}s</b></td></tr></tfoot>
        </table>
    </body>
    </html>
    """

    msg = MIMEMultipart()
    msg['From'] = GMAIL_USER
    msg['To'] = ", ".join(DESTINATION_EMAIL_LIST)
    msg['Subject'] = f"Pipeline Log: Maia Hidro Audit {datetime.date.today()}"
    msg.attach(MIMEText(html_body, 'html'))

    try:
        server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
        server.login(GMAIL_USER, GMAIL_APP_PASSWORD)
        server.sendmail(GMAIL_USER, DESTINATION_EMAIL_LIST, msg.as_string())
        server.quit()
        print("✅ Email enviado com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao enviar email: {e}")

# --- EXECUÇÃO ---
print("\n--- INICIO PIPELINE HIDROGRAFIA ---")
maia_geom = load_maia_boundary(MAIA_BOUNDARY_URL)

if maia_geom is not None:
    wfs = load_wfs_hidro(OFFICIAL_DATA_URL)
    osm = query_osm_waterways(maia_geom, API_TIMEOUT)

    df_res = reconcile_lines(wfs, osm)

    if not df_res.empty:
        url = save_sheet(df_res, GOOGLE_SHEET_NAME)
        context_vars['sheet_url'] = url

        m = create_map_hidro(df_res, maia_geom, HTML_MAP_FILENAME)
        print(f"✅ Mapa gerado: {HTML_MAP_FILENAME}")
        send_execution_email(url, df_res)
    else: print("❌ Tabela de resultados vazia.")
else: print("❌ Erro no Limite da Maia.")

Instalando bibliotecas...
Bibliotecas instaladas.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- INICIO PIPELINE HIDROGRAFIA ---
[Limite Maia] 1.24s
A carregar WFS Hidrografia...
[WFS Hidro (878 segmentos)] 1.37s
A carregar OSM (Waterways)...
[OSM Raw (252)] 34.92s
A reconciliar (WFS vs OSM)...
[Reconciliação Hidro] 2.68s
Gerando Mapa...
✅ Mapa gerado: /content/auditoria_hidro_2026-01-22.html
📧 A enviar email de relatório...
✅ Email enviado com sucesso!


In [ ]:
display(m)